# **1. Perkenalan Dataset**

## Indonesian Twitter Emotion Dataset

Dataset yang digunakan adalah **Indonesian Twitter Emotion Dataset**, sebuah kumpulan tweet berbahasa Indonesia yang telah dilabeli dengan kategori emosi.

### Sumber Dataset
- **Dataset 1**: EmoTweetID-Human — dataset tweet emosi Indonesia anotasi manusia
- **Dataset 2**: Twitter Emotion Dataset — dataset publik dari Kaggle ([link](https://www.kaggle.com/datasets/dennisherdi/indonesian-twitter-emotion))
- **Kamus Slang**: Kamus singkatan/slang bahasa Indonesia untuk normalisasi teks

### Deskripsi Dataset
Dataset ini digunakan untuk membangun sistem klasifikasi emosi pada platform konseling berbasis AI (**Myfess**). Model akan membantu mengidentifikasi kondisi emosional pengguna dari teks yang mereka tulis, sehingga konselor dan sistem dapat memberikan respons yang lebih tepat dan empatik.

### Kategori Emosi
Dataset mencakup beberapa kategori emosi utama dalam bahasa Indonesia, antara lain:
- **Anger** (Marah)
- **Fear** (Takut)
- **Happy** (Senang)
- **Love** (Cinta)
- **Sadness** (Sedih)
- **Surprise** (Kaget)

### Relevansi
Klasifikasi emosi dari teks sangat relevan untuk:
1. Aplikasi kesehatan mental dan konseling digital
2. Sistem deteksi dini kondisi psikologis pengguna
3. Personalisasi respons chatbot konseling


# **2. Import Library**

Mengimpor seluruh pustaka yang dibutuhkan untuk analisis data, preprocessing teks, dan persiapan pemodelan dengan IndoBERT.

In [ ]:
# Install dependencies yang dibutuhkan
!pip install -q transformers==4.41.2 accelerate==0.30.1 peft==0.11.1

: 

In [ ]:
# DATA MANIPULATION
import pandas as pd
import numpy as np
import json
import pickle
import re
import os

# VISUALIZATION
import matplotlib.pyplot as plt
import seaborn as sns

# MACHINE LEARNING
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)

# GOOGLE COLAB
from google.colab import drive

# KONFIGURASI VISUALISASI
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# RANDOM SEED untuk reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✓ Seluruh library berhasil dimuat")
print(f"  - pandas     : {pd.__version__}")
print(f"  - numpy      : {np.__version__}")
print(f"  - sklearn    : OK")

# **3. Memuat Dataset**

Memuat dua dataset tweet emosi bahasa Indonesia dan kamus slang dari Google Drive, kemudian menggabungkan keduanya menjadi satu dataset yang siap dianalisis.

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Definisi path dataset
PATH_DATASET_1  = '/content/drive/MyDrive/Project/Myfess/Datasets/EmoTweetID-Human.csv'
PATH_DATASET_2  = '/content/drive/MyDrive/Project/Myfess/Datasets/Twitter_Emotion_Dataset.csv'
PATH_SLANG_DICT = '/content/drive/MyDrive/Project/Myfess/Datasets/kamus_singkatan.csv'

In [ ]:
# === LOAD DATASET 1: EmoTweetID-Human ===
df1 = pd.read_csv(PATH_DATASET_1)
df1.columns = ['id', 'tweet', 'label']
df1 = df1[['tweet', 'label']]

# === LOAD DATASET 2: Twitter Emotion Dataset ===
df2 = pd.read_csv(PATH_DATASET_2, sep=None, engine='python')
df2.columns = ['label', 'tweet']
df2 = df2[['tweet', 'label']]

print(f"✓ Dataset 1 (EmoTweetID-Human) dimuat : {len(df1):,} baris")
print(f"✓ Dataset 2 (Twitter Emotion)  dimuat : {len(df2):,} baris")
print(f"\n--- Preview Dataset 1 ---")
print(df1.head(3))
print(f"\n--- Preview Dataset 2 ---")
print(df2.head(3))

In [ ]:
# === GABUNGKAN KEDUA DATASET ===
df = pd.concat([df1, df2], ignore_index=True)

# Hapus duplikat berdasarkan kolom tweet
df.drop_duplicates(subset=['tweet'], inplace=True)

# Standardisasi label: hapus whitespace dan ubah ke lowercase
df['label'] = df['label'].str.strip().str.lower()

# Reset index
df.reset_index(drop=True, inplace=True)

print(f"✓ Dataset gabungan : {len(df):,} baris unik")
print(f"✓ Missing values   : {df.isnull().sum().sum()}")
print(f"\nDistribusi Label:")
print(df['label'].value_counts())
print(f"\nInfo Dataset:")
df.info()

# **4. Exploratory Data Analysis (EDA)**

Memahami karakteristik dataset secara mendalam sebelum melakukan preprocessing, meliputi distribusi label, statistik teks, dan pola linguistik.

In [ ]:
# === 4.1 DISTRIBUSI KELAS EMOSI ===
emotion_counts = df['label'].value_counts()

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=emotion_counts.index, y=emotion_counts.values, palette='viridis')

for p in ax.patches:
    height = int(p.get_height())
    ax.annotate(f'{height:,}',
                xy=(p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title('Distribusi Emosi Dataset (Gabungan)', fontsize=16, fontweight='bold')
plt.xlabel('Kategori Emosi', fontsize=12)
plt.ylabel('Jumlah Tweet', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Total kategori emosi : {len(emotion_counts)}")
print(f"Kategori terbanyak   : {emotion_counts.index[0]} ({emotion_counts.iloc[0]:,} data)")
print(f"Kategori tersedikit  : {emotion_counts.index[-1]} ({emotion_counts.iloc[-1]:,} data)")
print(f"Rasio imbalance      : {emotion_counts.iloc[0]/emotion_counts.iloc[-1]:.2f}x")

In [ ]:
# === 4.2 STATISTIK PANJANG TEKS (SEBELUM PREPROCESSING) ===
df['raw_length'] = df['tweet'].apply(len)
df['word_count'] = df['tweet'].apply(lambda x: len(str(x).split()))

print("=" * 60)
print("STATISTIK PANJANG TEKS (RAW)")
print("=" * 60)
print(df[['raw_length', 'word_count']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram panjang karakter
axes[0].hist(df['raw_length'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df['raw_length'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["raw_length"].mean():.1f}')
axes[0].axvline(df['raw_length'].median(), color='green', linestyle='--',
                label=f'Median: {df["raw_length"].median():.1f}')
axes[0].set_title('Distribusi Panjang Tweet (Karakter)', fontweight='bold')
axes[0].set_xlabel('Jumlah Karakter')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Histogram jumlah kata
axes[1].hist(df['word_count'], bins=50, color='salmon', edgecolor='black', alpha=0.7)
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["word_count"].mean():.1f}')
axes[1].set_title('Distribusi Jumlah Kata per Tweet', fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# === 4.3 DISTRIBUSI PANJANG TEKS PER KATEGORI EMOSI ===
plt.figure(figsize=(12, 5))
df.boxplot(column='word_count', by='label', figsize=(12, 5))
plt.title('Distribusi Jumlah Kata per Kategori Emosi', fontsize=14, fontweight='bold')
plt.suptitle('')
plt.xlabel('Kategori Emosi')
plt.ylabel('Jumlah Kata')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Statistik per kategori
print("\nRata-rata jumlah kata per kategori emosi:")
print(df.groupby('label')['word_count'].agg(['mean', 'median', 'std']).round(2))

In [ ]:
# === 4.4 ANALISIS MISSING VALUES DAN DUPLIKAT ===
print("=" * 60)
print("ANALISIS KUALITAS DATA")
print("=" * 60)
print(f"Missing values per kolom:")
print(df[['tweet', 'label']].isnull().sum())
print(f"\nJumlah data duplikat (tweet): {df.duplicated(subset=['tweet']).sum()}")
print(f"\nContoh 5 baris pertama:")
print(df[['tweet', 'label']].head())

In [ ]:
# === 4.5 ANALISIS KARAKTER KHUSUS DALAM TEKS ===
# Deteksi pola yang perlu dibersihkan
has_url     = df['tweet'].str.contains(r'http\S+|www\S+', regex=True, na=False)
has_mention = df['tweet'].str.contains(r'@\w+', regex=True, na=False)
has_hashtag = df['tweet'].str.contains(r'#\w+', regex=True, na=False)
has_emoji   = df['tweet'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
has_number  = df['tweet'].str.contains(r'\d', regex=True, na=False)

noise_stats = {
    'Mengandung URL'     : has_url.sum(),
    'Mengandung Mention' : has_mention.sum(),
    'Mengandung Hashtag' : has_hashtag.sum(),
    'Mengandung Emoji'   : has_emoji.sum(),
    'Mengandung Angka'   : has_number.sum(),
}

print("=" * 60)
print("ANALISIS NOISE DALAM TEKS")
print("=" * 60)
for noise_type, count in noise_stats.items():
    pct = count / len(df) * 100
    print(f"  {noise_type:<25}: {count:,} ({pct:.1f}%)")

# Visualisasi noise
plt.figure(figsize=(10, 5))
plt.bar(noise_stats.keys(), noise_stats.values(), color='coral', edgecolor='black')
plt.title('Distribusi Noise dalam Dataset Tweet', fontsize=14, fontweight='bold')
plt.xlabel('Jenis Noise')
plt.ylabel('Jumlah Tweet')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Tahap preprocessing teks untuk membersihkan dan menormalisasi data tweet sebelum digunakan untuk fine-tuning IndoBERT. Proses ini mencakup:
1. Normalisasi kasus teks (lowercase)
2. Penghapusan URL, mention, dan hashtag
3. Normalisasi karakter berulang
4. Penghapusan karakter non-alfabet
5. Normalisasi slang/singkatan menggunakan kamus
6. Penghapusan kata tawa berlebihan
7. Label encoding dan train-test split

In [ ]:
# === 5.1 LOAD KAMUS SLANG ===
df_slang = pd.read_csv(PATH_SLANG_DICT, sep=None, engine='python', header=None)
slang_dict = dict(zip(df_slang[0], df_slang[1]))

print(f"✓ Kamus slang dimuat : {len(slang_dict):,} entri")
print(f"\nContoh normalisasi slang:")
sample_words = ['gw', 'lu', 'yg', 'bgt', 'tp', 'krn', 'dgn', 'utk']
for word in sample_words:
    normalized = slang_dict.get(word, word)
    if normalized != word:
        print(f"  '{word}' → '{normalized}'")

In [ ]:
# === 5.2 FUNGSI PREPROCESSING TEKS ===
def preprocess_text(text):
    """
    Membersihkan dan menormalisasi teks tweet bahasa Indonesia.

    Tahapan:
        1. Konversi ke lowercase
        2. Hapus URL, mention (@username), hashtag (#tag)
        3. Normalisasi karakter berulang (contoh: 'sediiihh' → 'sediih')
        4. Hapus karakter non-alfabet (angka, simbol, emoji)
        5. Normalisasi slang/singkatan menggunakan kamus
        6. Hapus kata tawa berlebihan (wk, ha, he, hi, ho)
        7. Hapus whitespace berlebih

    Args:
        text (str): Teks tweet mentah

    Returns:
        str: Teks yang sudah bersih dan ternormalisasi
    """
    # Step 1: Konversi ke string dan lowercase
    text = str(text).lower()

    # Step 2: Hapus URL, mention, dan hashtag
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#\w+', '', text)

    # Step 3: Normalisasi karakter berulang (maks 2 huruf berulang)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Step 4: Hapus karakter non-alfabet
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Step 5: Normalisasi slang menggunakan kamus
    words = text.split()
    normalized_words = [slang_dict.get(word, word) for word in words]
    text = " ".join(normalized_words)

    # Step 6: Hapus kata tawa berlebihan
    text = re.sub(r'\b(wk|ha|he|hi|ho)+\b', '', text, flags=re.IGNORECASE)

    # Step 7: Hapus whitespace berlebih dan trim
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Test fungsi preprocessing
sample_tweets = [
    "Aku sediiihhhh bgttt 😭😭 gara2 nilai matematika jelek bgt huhuhu",
    "@siswa123 Wkwkwk jgn sedih dong!! yg semangat yaa #motivasi",
    "https://example.com gw stress bgt sm tugas akhir yg banyaaakkk bgttt"
]

print("Contoh Hasil Preprocessing:\n")
for i, tweet in enumerate(sample_tweets, 1):
    cleaned = preprocess_text(tweet)
    print(f"Tweet {i}:")
    print(f"  Original : {tweet}")
    print(f"  Cleaned  : {cleaned}\n")

In [ ]:
# === 5.3 TERAPKAN PREPROCESSING KE SELURUH DATASET ===
df['clean_tweet'] = df['tweet'].apply(preprocess_text)

# Hapus baris dengan teks kosong setelah preprocessing
initial_count = len(df)
df = df[df['clean_tweet'].str.strip() != '']
df.reset_index(drop=True, inplace=True)
final_count = len(df)

print(f"✓ Preprocessing selesai")
print(f"  - Data awal   : {initial_count:,} baris")
print(f"  - Data dihapus: {initial_count - final_count:,} baris (teks kosong)")
print(f"  - Data final  : {final_count:,} baris")

print("\n" + "="*80)
print("SAMPLE HASIL PREPROCESSING")
print("="*80)
print(df[['tweet', 'clean_tweet', 'label']].head(5).to_string())

In [ ]:
# === 5.4 ANALISIS PANJANG TEKS SETELAH PREPROCESSING ===
df['text_length'] = df['clean_tweet'].apply(len)
df['clean_word_count'] = df['clean_tweet'].apply(lambda x: len(x.split()))

print("Statistik Panjang Teks Setelah Preprocessing:")
print(df[['text_length', 'clean_word_count']].describe().round(2))

# Visualisasi perbandingan sebelum vs sesudah
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df['raw_length'],   bins=50, alpha=0.6, color='skyblue',  label='Sebelum', edgecolor='black')
axes[0].hist(df['text_length'],  bins=50, alpha=0.6, color='salmon',   label='Sesudah', edgecolor='black')
axes[0].set_title('Perbandingan Panjang Teks (Karakter)', fontweight='bold')
axes[0].set_xlabel('Jumlah Karakter')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

axes[1].hist(df['word_count'],       bins=50, alpha=0.6, color='skyblue', label='Sebelum', edgecolor='black')
axes[1].hist(df['clean_word_count'], bins=50, alpha=0.6, color='salmon',  label='Sesudah', edgecolor='black')
axes[1].set_title('Perbandingan Jumlah Kata', fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()

# Rekomendasi max_length untuk tokenizer
percentile_95 = np.percentile(df['text_length'], 95)
print(f"\n📊 Rekomendasi max_length untuk IndoBERT tokenizer:")
print(f"   95% data memiliki panjang ≤ {percentile_95:.0f} karakter")
print(f"   → Disarankan menggunakan max_length = 128 token")

In [ ]:
# === 5.5 LABEL ENCODING ===
df['label'] = df['label'].str.strip().str.lower()

le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

label_mapping = dict(zip(le.classes_, le.transform(le.classes_).tolist()))
num_labels = len(le.classes_)

print("=" * 60)
print("LABEL ENCODING")
print("=" * 60)
print(f"Jumlah kategori emosi: {num_labels}")
print(f"\nMapping Label → ID:")
for label, idx in sorted(label_mapping.items(), key=lambda x: x[1]):
    count = len(df[df['label'] == label])
    print(f"  {idx} → {label:<12} ({count:,} data)")

In [ ]:
# === 5.6 TRAIN-TEST SPLIT ===
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df['label_id']
)

df_train.reset_index(drop=True, inplace=True)
df_val.reset_index(drop=True, inplace=True)

print(f"✓ Data Training   : {len(df_train):,} baris ({len(df_train)/len(df)*100:.1f}%)")
print(f"✓ Data Validation : {len(df_val):,} baris ({len(df_val)/len(df)*100:.1f}%)")

# Verifikasi distribusi stratified
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_train['label'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', title='Distribusi Label - Training Set')
df_val['label'].value_counts().plot(kind='bar',   ax=axes[1], color='coral',     title='Distribusi Label - Validation Set')
for ax in axes:
    ax.set_xlabel('Kategori Emosi')
    ax.set_ylabel('Jumlah Tweet')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# === 5.7 SIMPAN DATASET HASIL PREPROCESSING ===
OUTPUT_DIR = './twitter_emotion_preprocessing'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Simpan dataset
df_train[['clean_tweet', 'label', 'label_id']].to_csv(f'{OUTPUT_DIR}/train.csv', index=False)
df_val[['clean_tweet',   'label', 'label_id']].to_csv(f'{OUTPUT_DIR}/val.csv',   index=False)
df[['tweet', 'clean_tweet', 'label', 'label_id']].to_csv(f'{OUTPUT_DIR}/full.csv', index=False)

# Simpan label encoder dan metadata
with open(f'{OUTPUT_DIR}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

metadata = {
    'num_labels'    : num_labels,
    'label_mapping' : label_mapping,
    'total_samples' : len(df),
    'train_samples' : len(df_train),
    'val_samples'   : len(df_val),
    'max_length'    : 128
}
with open(f'{OUTPUT_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Dataset preprocessing berhasil disimpan di: {OUTPUT_DIR}/")
print(f"  - train.csv          ({len(df_train):,} baris)")
print(f"  - val.csv            ({len(df_val):,} baris)")
print(f"  - full.csv           ({len(df):,} baris)")
print(f"  - label_encoder.pkl")
print(f"  - metadata.json")
print("\n✅ Eksperimen selesai! Dataset siap untuk fine-tuning IndoBERT.")